# NASA CMAPSS — Feature Engineering
## Complete pipeline: drop constant sensors → normalize → rolling features

This notebook:
1. Loads `data/raw/train.csv` and `data/raw/test.csv` from notebook 01_data_preprocessing_RUL
2. **Step 1** — Drops constant/near-constant sensors (zero degradation signal)
3. **Step 2** — Normalizes sensors per operating condition cluster (prevents confusing altitude shifts with degradation)
4. **Step 3** — Adds rolling features (mean + std over 5 and 10 cycle windows) per engine per sensor
5. Saves `data/processed/train.csv` and `data/processed/test.csv`

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

# ── Constants ──────────────────────────────────────────────────────────────────
STD_THRESHOLD        = 0.1          # sensors with std below this are dropped
N_OP_CLUSTERS        = 6            # number of operating condition clusters
ROLLING_WINDOWS      = [5, 10]      # cycle windows for rolling statistics
RANDOM_SEED          = 42

# Literature-confirmed constant sensors across all FD00X subsets
KNOWN_CONSTANT_SENSORS = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']
SENSOR_COLS            = [f's{i}' for i in range(1, 22)]
OP_COLS                = ['op1', 'op2', 'op3']
NON_SENSOR_COLS        = ['engine_id', 'cycle', 'op1', 'op2', 'op3', 'dataset_id', 'RUL']

os.makedirs('data/processed', exist_ok=True)
print('Libraries loaded. Constants set.')

## Load Raw Data

In [ ]:
train = pd.read_csv('../data/raw/train.csv')
test  = pd.read_csv('../data/raw/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Columns: {list(train.columns)}')
train.head()

---
# Step 1: Drop Constant / Low-Variance Sensors

Sensors with near-zero variance carry no degradation signal — including them only adds noise.
We check variance **per dataset_id** to avoid discarding sensors that matter in FD002/FD004
but happen to look flat in FD001.

Drop rule: sensor must be constant (std < threshold) in **all 4 subsets** to be dropped.
We then take the **union** with the literature-confirmed list as a safety net.

In [ ]:
# Global std overview
global_std = train[SENSOR_COLS].std().sort_values()
print('=== Global Sensor Standard Deviations (ascending) ===')
print(global_std.to_string())

print(f'\nSensors with global std < {STD_THRESHOLD}:')
print(global_std[global_std < STD_THRESHOLD].index.tolist())

In [ ]:
# Per-dataset std — the more reliable signal
per_dataset_std = train.groupby('dataset_id')[SENSOR_COLS].std()
is_constant     = per_dataset_std < STD_THRESHOLD
constant_in_all = is_constant.all(axis=0)

print('=== Per-Dataset Standard Deviations ===')
print(per_dataset_std.T.to_string())
print(f'\nConstant in ALL subsets: {constant_in_all[constant_in_all].index.tolist()}')

In [ ]:
# Final drop list = data-driven union literature-confirmed
data_driven_drops = constant_in_all[constant_in_all].index.tolist()
sensors_to_drop   = sorted(set(data_driven_drops) | set(KNOWN_CONSTANT_SENSORS))
sensors_to_keep   = [s for s in SENSOR_COLS if s not in sensors_to_drop]

print(f'Sensors DROPPED ({len(sensors_to_drop)}): {sensors_to_drop}')
print(f'Sensors KEPT   ({len(sensors_to_keep)}): {sensors_to_keep}')

In [ ]:
# Apply the drop
retain_cols = NON_SENSOR_COLS + sensors_to_keep
train = train[retain_cols].copy()
test  = test[retain_cols].copy()

print(f'Train after drop: {train.shape}')
print(f'Test  after drop: {test.shape}')

---
# Step 2: Normalize Sensors per Operating Condition Cluster

**Why not normalize globally?**
FD002/FD004 have 6 operating conditions (altitude, Mach, throttle). The same sensor
reads very differently at sea level vs. cruise altitude — not because the engine is
degrading, but because the physics change. Global normalization would confuse
operating condition shifts with degradation trends.

**Strategy:**
1. Cluster `op1/op2/op3` into 6 clusters using KMeans — fit on train only
2. Apply the same cluster assignment to test using `.predict()` — no refit
3. Fit a MinMaxScaler per `(dataset_id, op_cluster)` group on train
4. Transform both train and test with that same scaler — no data leakage

In [ ]:
# Cluster operating conditions — fit on train only
kmeans = KMeans(n_clusters=N_OP_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
train['op_cluster'] = kmeans.fit_predict(train[OP_COLS])
test['op_cluster']  = kmeans.predict(test[OP_COLS])    # same model, no refit

print('Operating condition cluster distribution (train):')
print(train.groupby('dataset_id')['op_cluster'].value_counts().sort_index().to_string())

In [ ]:
# Normalize per (dataset_id, op_cluster) — fit on train, transform both
scalers = {}

for (dataset_id, op_cluster), train_idx in train.groupby(['dataset_id', 'op_cluster']).groups.items():
    scaler = MinMaxScaler()

    # Fit and transform training rows for this group
    train.loc[train_idx, sensors_to_keep] = scaler.fit_transform(
        train.loc[train_idx, sensors_to_keep]
    )

    # Transform matching test rows with the SAME scaler (no refit)
    test_idx = test.index[
        (test['dataset_id'] == dataset_id) &
        (test['op_cluster'] == op_cluster)
    ]
    if len(test_idx) > 0:
        test.loc[test_idx, sensors_to_keep] = scaler.transform(
            test.loc[test_idx, sensors_to_keep]
        )

    scalers[(dataset_id, op_cluster)] = scaler

print(f'Normalization complete. Scalers fitted: {len(scalers)}')
print(f'\nSensor value ranges after normalization (train) — all should be in [0, 1]:')
print(train[sensors_to_keep].agg(['min', 'max']).T.to_string())

---
# Step 3: Rolling Features

A single row's sensor value tells you very little — what matters is the **trend**.
Rolling statistics capture how each sensor is changing over recent cycles.

For each kept sensor and each window size (5 and 10 cycles), we compute:
- **Rolling mean** — smoothed signal, reveals trend direction
- **Rolling std** — captures volatility / instability as the engine degrades

Rolling is computed **within each engine** (grouped by `engine_id`) so cycle 1 of
engine 2 does not contaminate engine 1's window.

`min_periods=1` ensures no NaN values for early cycles where the full window is not available yet.

In [ ]:
def add_rolling_features(df, sensor_cols, windows, group_col='engine_id'):
    """Add rolling mean and std for each sensor over given windows, grouped by engine."""
    new_cols = {}
    for window in windows:
        for sensor in sensor_cols:
            grouped = df.groupby(group_col)[sensor]
            new_cols[f'{sensor}_rmean_{window}'] = grouped.transform(
                lambda x: x.rolling(window, min_periods=1).mean()
            )
            new_cols[f'{sensor}_rstd_{window}'] = grouped.transform(
                lambda x: x.rolling(window, min_periods=1).std().fillna(0)
            )
    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)


print('Adding rolling features to train...')
train = add_rolling_features(train, sensors_to_keep, ROLLING_WINDOWS)

print('Adding rolling features to test...')
test  = add_rolling_features(test,  sensors_to_keep, ROLLING_WINDOWS)

n_new = len(sensors_to_keep) * len(ROLLING_WINDOWS) * 2
print(f'\nAdded {n_new} rolling feature columns')
print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')

rolling_cols = [c for c in train.columns if '_rmean_' in c or '_rstd_' in c]
print(f'\nRolling feature columns ({len(rolling_cols)} total):')
print(rolling_cols)

In [ ]:
# Sanity check: rolling features for one engine should make visual sense
sample_engine = train['engine_id'].iloc[0]
cols_to_show  = ['engine_id', 'cycle', 's2', 's2_rmean_5', 's2_rstd_5', 's2_rmean_10', 's2_rstd_10']

print(f'Rolling feature sample — Engine {sample_engine}, first 15 cycles:')
print(train[train['engine_id'] == sample_engine][cols_to_show].head(15).to_string(index=False))

## Save to CSV

In [ ]:
train.to_csv('../data/processed/train.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)
print('Saved: ../data/processed/train.csv')
print('Saved: ../data/processed/test.csv')

## Verification

In [ ]:
print('=== VERIFICATION ===\n')

# 1. Dropped sensors must be absent
for s in sensors_to_drop:
    assert s not in train.columns, f'ERROR: {s} still present in train!'
    assert s not in test.columns,  f'ERROR: {s} still present in test!'
print(f'[PASS] All {len(sensors_to_drop)} dropped sensors are absent')

# 2. No NaN values introduced anywhere
assert train.isnull().sum().sum() == 0, 'ERROR: NaNs found in train!'
assert test.isnull().sum().sum()  == 0, 'ERROR: NaNs found in test!'
print('[PASS] No NaN values in train or test')

# 3. Normalized sensor values should be in [0, 1] for train
sensor_min = train[sensors_to_keep].min().min()
sensor_max = train[sensors_to_keep].max().max()
assert round(sensor_min, 6) >= 0.0, f'ERROR: Sensor min below 0: {sensor_min}'
assert round(sensor_max, 6) <= 1.0, f'ERROR: Sensor max above 1: {sensor_max}'
print('[PASS] All sensor values in [0, 1] after normalization')

# 4. Rolling feature columns must all exist
for window in ROLLING_WINDOWS:
    for sensor in sensors_to_keep:
        assert f'{sensor}_rmean_{window}' in train.columns, f'Missing: {sensor}_rmean_{window}'
        assert f'{sensor}_rstd_{window}'  in train.columns, f'Missing: {sensor}_rstd_{window}'
print(f'[PASS] All {n_new} rolling feature columns present')

# 5. Rolling std at cycle 1 of every engine must be 0 (only one observation in window)
first_cycles = train[train['cycle'] == train.groupby('engine_id')['cycle'].transform('min')]
rstd_cols    = [c for c in train.columns if '_rstd_' in c]
assert (first_cycles[rstd_cols] == 0).all().all(), 'ERROR: Rolling std at first cycle is not 0!'
print('[PASS] Rolling std is 0 at first cycle of each engine (correct)')

# 6. RUL values must be unchanged from raw input
train_raw = pd.read_csv('../data/raw/train.csv')
assert train['RUL'].reset_index(drop=True).equals(
    train_raw['RUL'].reset_index(drop=True)
), 'ERROR: RUL values changed!'
print('[PASS] RUL values unchanged')

print(f'\nFinal summary:')
print(f'  Sensors kept:    {sensors_to_keep}')
print(f'  Sensors dropped: {sensors_to_drop}')
print(f'  Rolling windows: {ROLLING_WINDOWS} cycles (mean + std each)')
print(f'  Train shape:     {train.shape}')
print(f'  Test shape:      {test.shape}')
print(f'  Total features:  {train.shape[1]} columns')

In [ ]:
print('\nFeature engineering complete! data/processed/ is ready for modeling.')